# Pose Analysis Workflow

Pull **everything** out of a set of Bonsai/HARP sessions — including the
DeepLabCut **pose** data in each session's `DLC/` folder — time-align each
session, then stack every data type across sessions into one container:

```
select_sessions -> filter_sessions -> build_sessions -> combine_sessions
 (which sessions)   (narrow by day/    (read+align each)  (stack each type
                     name, require DLC)                    across sessions)
```

The end result is

```
selected_sessions = { 'dlc:position': <DataArray>, 'dlc:confidence': <DataArray>,
                      'events': <DataFrame>, 'nosepoke:Activations': <DataArray>, ... }
```

— one entry per data type, each **concatenated across sessions** and carrying a
`label` coord/column that tags every row/sample with its source session.

**Pose alignment.** DeepLabCut writes one row per video frame with no clock of
its own. The Bonsai `VideoData` CSV logs one row per frame too, and its
`Seconds` column is the HARP timestamp of each frame — the *same* clock as
ExperimentEvents. So pose is aligned by a positional join of those frame times
onto the DLC rows (handled inside `DLCPose`); no TTL conversion is needed. If a
session split its recording into segments, each DLC file is paired with the
`VideoData` file of matching frame count and the segments are concatenated in
time order (a length mismatch is surfaced as a warning).

## 0. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

# --- data-conduit (editable install now; identical import after the pip release) ---
from data_conduit.sessiongroups import (
    select_sessions,
    default_harp_catalog,
    DataStructureSpec,
    load_session,
    build_sessions,
    combine_sessions,
    add_label_column,
)
from data_conduit.datasources.pose import DLCPose, pose_to_movement

# --- workspace-local helper (this notebook's folder) ---
sys.path.insert(0, str(Path.cwd()))
from pose_analysis_utils import filter_sessions, session_path_string

# Optional: interactive tables.
try:
    import itables
    itables.init_notebook_mode()
    itables.options.maxRows = 100
except ModuleNotFoundError:
    itables = None

warnings.filterwarnings("ignore")  # quieten the preset loaders' verbose hints

### 0.1 Config — paths & selection knobs

`DATA_ROOT` is the top-level Bonsai output. The pose-bearing sessions live under
`<DATA_ROOT>/<PHASE>/<MOUSE_ID>/<day>/<session>`, so we point `DATA_DIR` at one
phase+mouse and let `select_sessions` find the day/session folders below it.

In [ ]:
# ============================ EDIT THESE ============================
DATA_ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput")
PHASE     = "Shaping"            # Shaping / Training / Test
MOUSE_ID  = "FbR_M01569522"      # mouse folder under <DATA_ROOT>/<PHASE>
DATA_DIR  = DATA_ROOT / PHASE / MOUSE_ID

# Which sessions to keep. Matched against each session's "{day}/{name}" string.
#   None                      -> every session under DATA_DIR
#   "Day1/"                   -> exactly Day1 (trailing / excludes Day10..Day19)
#   r"^Day(1|2)/"  (regex=True)-> Day1 and Day2
SESSION_PATTERN = None
SESSION_REGEX   = False
REQUIRE_DLC     = True           # drop sessions with no DLC/ folder

# HARP device YAMLs (gitignored; needed ONLY for nosepoke/soundcard/camera).
# If absent, those streams are skipped and you still get events + video + pose.
def _repo_root():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "pyproject.toml").exists():
            return p
    return Path.cwd()

REPO_ROOT      = _repo_root()
DEVICE_YAML    = REPO_ROOT / "device.yml"
SOUNDCARD_YAML = REPO_ROOT / "soundcard.yml"
# ===================================================================

assert DATA_DIR.is_dir(), f"DATA_DIR not found: {DATA_DIR}"
print("data dir       :", DATA_DIR)
print("device yaml    :", DEVICE_YAML, "(found)" if DEVICE_YAML.exists() else "(MISSING -> nosepoke/soundcard skipped)")
print("soundcard yaml :", SOUNDCARD_YAML, "(found)" if SOUNDCARD_YAML.exists() else "(MISSING -> soundcard skipped)")

## 1. Select sessions

`select_sessions` walks `DATA_DIR` to the session folders (`depth=1`: `day/session`)
and records the `day` as metadata on each. `filter_sessions` then narrows by a
substring/regex on the flattened `"{day}/{session}"` string and (here) keeps only
sessions that actually contain a `DLC/` folder — important before the
cross-session combine, whose default `on="intersection"` would otherwise drop
pose if any session lacked it.

In [ ]:
sessions = select_sessions(DATA_DIR, depth=1, level_names=("day",))
print(f"found {len(sessions)} sessions under {PHASE}/{MOUSE_ID}")

sessions = filter_sessions(
    sessions,
    SESSION_PATTERN,
    regex=SESSION_REGEX,
    require_dirs=("DLC",) if REQUIRE_DLC else None,
)
print(f"kept  {len(sessions)} after filtering")

selection_table = pd.DataFrame(
    [
        {"session": n, **{k: v for k, v in e.items() if k != "path"},
         "has_DLC": (e["path"] / "DLC").is_dir()}
        for n, e in sessions.items()
    ]
)
display(selection_table.head(20))

## 2. Catalog — what to extract

The catalog lists the data structures to read from each session. We start from
the lab's default HARP set and add a **DeepLabCut pose** spec. The pose spec is
*same-clock* (`sync=None`): `DLCPose` attaches each frame's `VideoData`
timestamp, so pose lands on the shared session clock and needs no TTL step.

This dataset logs camera metadata as `VideoData` CSVs rather than a
`Camera0Frames` HARP folder, so we drop the `camera` spec.

In [ ]:
catalog = default_harp_catalog(
    device_yaml=DEVICE_YAML,
    soundcard_yaml=SOUNDCARD_YAML,
    include_video=True,
    include_session_settings=True,
)
catalog.remove("camera")  # camera metadata comes from VideoData here, not Camera0Frames

# Add DeepLabCut pose -> members 'dlc:position' and 'dlc:confidence'.
# on_length_mismatch="warn": a session whose DLC/VideoData frame counts disagree
# warns (a likely dropped-frame / wrong-file discrepancy) instead of aborting the
# whole batch. Use "error" for a strict single-session check.
catalog.add(DataStructureSpec(
    name="dlc",
    reader=lambda p: DLCPose(experiment_directory_path=p, on_length_mismatch="warn"),
    required=False,
))

print("will extract:", [s.name for s in catalog.enabled_specs()])

## 3. One session — sanity-check the pose alignment

Load a single session and confirm pose shares the events clock: the pose `Time`
axis (camera frame `Seconds`) spans the same window as the events log.

In [ ]:
one_path = next(iter(sessions.values()))["path"]
session = load_session(one_path, catalog, verbose=False)
print("session:", one_path.name)
print("members:", session.names)

position = session.data["dlc:position"]
events = session.data["events"]
print()
print("events Time:", round(float(events.index.min()), 2), "->", round(float(events.index.max()), 2), "s")
print("pose   Time:", round(float(position["Time"].min()), 2), "->", round(float(position["Time"].max()), 2), "s")
print("  (pose frame times come from VideoData 'Seconds' -> same clock as events)")
position

## 4. Build every session, then combine

`build_sessions` runs the load+align pipeline on every selected session
(`normalise=False` — times stay on their shared original axis). `combine_sessions`
with `type_name=None` stacks **every** member across the group and returns one
`Session` whose `.data` is the container we want.

In [ ]:
group = build_sessions(
    sessions,
    catalog,
    label=f"{PHASE}/{MOUSE_ID}",
    sort_by=("day", "label"),
    normalise=False,
)
print(f"built {len(group)} sessions")
print("members each provides:", group[0].names)

In [ ]:
combined = combine_sessions(group, type_name=None)   # whole-session combine -> Session
selected_sessions = combined.data                    # {key: concatenated obj w/ session label}

print("selected_sessions keys:")
for key, obj in selected_sessions.items():
    shape = dict(obj.sizes) if hasattr(obj, "sizes") else f"{len(obj)} rows"
    print(f"  {key:22s} {type(obj).__name__:11s} {shape}")

### 4.1 Visualise the combined data structure

A quick look at what `selected_sessions` holds: one row per member with its type,
shape, how many sessions are stacked into it, and its extra coords/columns. Below
the table, the rich xarray view of `dlc:position` shows the `Time × keypoints ×
space` layout and the per-frame `label` coord that names each sample's source
session.

In [ ]:
# One row per member: type, shape, sessions stacked, and extra coords/columns.
rows = []
for key, obj in selected_sessions.items():
    if hasattr(obj, "sizes"):                                  # xarray DataArray / Dataset
        shape = " × ".join(f"{d}:{n}" for d, n in obj.sizes.items())
        n_sessions = len(set(obj["label"].values)) if "label" in obj.coords else None
        extras = [c for c in obj.coords if c not in obj.dims]
    else:                                                      # pandas DataFrame
        shape = f"rows:{len(obj)}"
        n_sessions = int(obj["label"].nunique()) if "label" in obj.columns else None
        extras = list(obj.columns)[:8]
    rows.append({"member": key, "type": type(obj).__name__, "shape": shape,
                 "n_sessions": n_sessions, "extra coords/cols": ", ".join(map(str, extras))})

structure = pd.DataFrame(rows).set_index("member")
display(structure)

# Rich xarray view of one member: dims, coords (incl. the per-frame `label`), attrs.
selected_sessions["dlc:position"]

## 5. Work with the combined pose

`selected_sessions['dlc:position']` is one `(Time x keypoints x space)` DataArray
spanning every session, with a `label` coord on `Time` naming each sample's
source session.

In [ ]:
position = selected_sessions["dlc:position"]      # (Time x keypoints x space)
confidence = selected_sessions["dlc:confidence"]  # (Time x keypoints)

print("combined position:", dict(position.sizes))
print("sessions tagged  :", len(set(position["label"].values)))
print()
print("frames per session:")
display(pd.Series(position["label"].values).value_counts().sort_index().to_frame("frames"))

In [ ]:
# Pull one session's nose track out via the label coord.
first_label = str(position["label"].values[0])
nose_first = position.sel(keypoints="nose").where(position["label"] == first_label, drop=True)
print(f"nose track for {first_label}: {dict(nose_first.sizes)}")
nose_first.isel(Time=slice(0, 5))

### 5.1 Tag samples with a derived grouping

`add_label_column` attaches a coord computed per sample — here the `day` each
sample came from, mapped back from the selection metadata.

In [ ]:
label_to_day = {name: entry["day"] for name, entry in sessions.items()}
position_with_day = add_label_column(
    position,
    "day",
    lambda da: np.array([label_to_day[lab] for lab in da["label"].values]),
    dim="Time",
)
print("frames per day:")
display(pd.Series(position_with_day["day"].values).value_counts().sort_index().to_frame("frames"))

## 6. Example: analysing pose with `movement`

[`movement`](https://movement.neuroinformatics.dev/) analyses one recording on a
monotonic time axis, so here we work on **individual sessions** (not the
cross-session stack, whose time resets at each session boundary). Choose which to
look at by editing the `MOVEMENT_SESSIONS` **list** — add/remove session keys and
re-run. For each session we:

1. build a `movement` poses dataset with `pose_to_movement`,
2. drop low-confidence detections with `filter_by_confidence`,
3. compute keypoint speed with `compute_speed`, and
4. plot the trajectory (coloured by time) and the speed trace.

In [ ]:
import matplotlib.pyplot as plt
from data_conduit.datasources.pose import read_dlc_pose, pose_to_movement
from movement.filtering import filter_by_confidence
from movement.kinematics import compute_speed

# ---- EDIT: which sessions to inspect (keys from `sessions`), and the knobs ----
MOVEMENT_SESSIONS = list(sessions)[:2]     # add/remove session keys here
KEYPOINT          = "body"                 # bodypart to plot
CONFIDENCE_MIN    = 0.9                    # blank detections below this likelihood
# ------------------------------------------------------------------------------

fig, axes = plt.subplots(len(MOVEMENT_SESSIONS), 2,
                         figsize=(12, 4.2 * len(MOVEMENT_SESSIONS)), squeeze=False)

for r, key in enumerate(MOVEMENT_SESSIONS):
    path = sessions[key]["path"]

    # 1) read this session's pose, convert to a movement dataset (time in seconds).
    pose = read_dlc_pose(path, on_length_mismatch="warn")
    fps = 1.0 / float(np.median(np.diff(pose["position"]["Time"].values)))
    ds = pose_to_movement(pose["position"], pose["confidence"], fps=fps)

    # 2) movement: blank low-confidence detections; 3) movement: keypoint speed.
    clean = filter_by_confidence(ds["position"], ds["confidence"],
                                 threshold=CONFIDENCE_MIN, print_report=False)
    speed = compute_speed(clean).sel(keypoints=KEYPOINT).squeeze()

    # 4) plot trajectory (x/y coloured by time) + speed over time.
    kp = clean.sel(individuals="individual_0", keypoints=KEYPOINT)
    x, y, t = kp.sel(space="x").values, kp.sel(space="y").values, kp["time"].values

    ax0 = axes[r][0]
    sc = ax0.scatter(x, y, c=t, s=2, cmap="viridis")
    ax0.set(title=f"{key}\n{KEYPOINT} trajectory", xlabel="x (px)", ylabel="y (px)")
    ax0.invert_yaxis(); ax0.set_aspect("equal", "box")
    fig.colorbar(sc, ax=ax0, label="time (s)")

    ax1 = axes[r][1]
    ax1.plot(speed["time"].values, speed.values, lw=0.6)
    ax1.set(title=f"{key}\n{KEYPOINT} speed  (~{fps:.0f} fps)",
            xlabel="time (s)", ylabel="speed (px/s)")

fig.tight_layout()
plt.show()
print(f"analysed {len(MOVEMENT_SESSIONS)} session(s): {MOVEMENT_SESSIONS}")

### 6.1 Export the whole combined stack to `movement`

`pose_to_movement` also works on the **combined** object from section 4 — useful
for saving/handing off every session at once. Kinematics across session
boundaries aren't meaningful, but the dataset (with its per-frame `label` coord)
is valid for export and per-session grouping.

In [ ]:
poses_ds = pose_to_movement(position, confidence)   # combined: dims (time, individuals, keypoints, space)
poses_ds

## Notes

- **HARP YAMLs.** `device.yml` / `soundcard.yml` are gitignored and live at the
  repo root by convention. Without them, `nosepoke` / `soundcard` simply skip
  (they're `required=False`) and you still get `events`, `video`, and the DLC
  `dlc:position` / `dlc:confidence`. Drop the YAMLs in to enable the HARP streams.
- **Member keys** use the `spec:member` convention (`dlc:position`,
  `nosepoke:Activations`, …); `events` / `video` are bare DataFrames.
- **`normalise`** is off: every stream keeps its shared session clock. Pass
  `normalise=True` to `build_sessions` to shift each session to start at t=0.
- **Combine scope.** `combine_sessions(type_name=None)` uses
  `on="intersection"`, keeping members present in *every* session. Requiring a
  `DLC/` folder in section 1 keeps pose in the intersection. To stack just one
  type across whatever sessions have it, call e.g.
  `combine_sessions(group, type_name="dlc:position")`.
- **Length validation.** A DLC file whose frame count matches no `VideoData`
  segment is flagged (warn here; set `on_length_mismatch="error"` in the spec to
  make it fatal) — a likely dropped-frame or wrong/stale-file discrepancy.